# EXP-006 | CatBoost vs XGBoost 비교

EXP-003 동일 피처로 CatBoost, XGBoost 성능 비교.

| 항목 | 내용 |
|---|---|
| 기반 실험 | EXP-003 피처 |
| 모델 | CatBoost / XGBoost |
| CV | Stratified 5-Fold |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score, recall_score, precision_score, accuracy_score
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']   = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부'] = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 85)  /  X_test: (90067, 85)


## 2. CatBoost

In [3]:
EXP_NO     = 6
MODEL_NAME = 'CatBoost'

cat_params = dict(
    iterations       = 2000,
    learning_rate    = 0.05,
    depth            = 6,
    l2_leaf_reg      = 3,
    loss_function    = 'Logloss',
    eval_metric      = 'AUC',
    auto_class_weights = 'Balanced',
    random_seed      = SEED,
    verbose          = False,
    early_stopping_rounds = 100,
)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_cat    = np.zeros(len(X_train))
test_cat   = np.zeros(len(X_test))
fold_aucs  = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = CatBoostClassifier(**cat_params)
    model.fit(X_tr, y_tr,
              eval_set=Pool(X_val, y_val),
              use_best_model=True)

    val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    fold_aucs.append(auc)
    oof_cat[val_idx] = val_prob
    test_cat        += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold}  best_iter={model.best_iteration_:4d}  AUC={auc:.5f}')

oof_auc_cat   = roc_auc_score(y_train, oof_cat)
oof_prauc_cat = average_precision_score(y_train, oof_cat)
oof_f1_cat    = f1_score(y_train, (oof_cat >= 0.5).astype(int))
print(f'\nCatBoost OOF ROC-AUC: {oof_auc_cat:.5f}')
print(f'CatBoost OOF PR-AUC : {oof_prauc_cat:.5f}')
print(f'CatBoost OOF F1     : {oof_f1_cat:.5f}  (threshold=0.5)')
print(f'Fold 평균           : {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}')

  Fold 1  best_iter= 394  AUC=0.73805
  Fold 2  best_iter= 450  AUC=0.74299
  Fold 3  best_iter= 432  AUC=0.73996
  Fold 4  best_iter= 394  AUC=0.73843
  Fold 5  best_iter= 493  AUC=0.74073

CatBoost OOF ROC-AUC: 0.74003
CatBoost OOF PR-AUC : 0.45086
CatBoost OOF F1     : 0.51729  (threshold=0.5)
Fold 평균           : 0.74003 ± 0.00177


## 3. XGBoost

In [4]:
EXP_NO     = 7
MODEL_NAME = 'XGBoost'

neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb_params = dict(
    n_estimators      = 2000,
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_weight  = 50,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    scale_pos_weight  = neg_pos_ratio,
    eval_metric       = 'auc',
    tree_method       = 'hist',
    random_state      = SEED,
    verbosity         = 0,
    early_stopping_rounds = 100,
)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_xgb    = np.zeros(len(X_train))
test_xgb   = np.zeros(len(X_test))
fold_aucs  = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)

    val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    fold_aucs.append(auc)
    oof_xgb[val_idx] = val_prob
    test_xgb        += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold}  best_iter={model.best_iteration:4d}  AUC={auc:.5f}')

oof_auc_xgb   = roc_auc_score(y_train, oof_xgb)
oof_prauc_xgb = average_precision_score(y_train, oof_xgb)
oof_f1_xgb    = f1_score(y_train, (oof_xgb >= 0.5).astype(int))
print(f'\nXGBoost OOF ROC-AUC: {oof_auc_xgb:.5f}')
print(f'XGBoost OOF PR-AUC : {oof_prauc_xgb:.5f}')
print(f'XGBoost OOF F1     : {oof_f1_xgb:.5f}  (threshold=0.5)')
print(f'Fold 평균          : {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}')

  Fold 1  best_iter= 252  AUC=0.73805
  Fold 2  best_iter= 246  AUC=0.74315
  Fold 3  best_iter= 179  AUC=0.74021
  Fold 4  best_iter= 190  AUC=0.73842
  Fold 5  best_iter= 239  AUC=0.74104

XGBoost OOF ROC-AUC: 0.74016
XGBoost OOF PR-AUC : 0.45022
XGBoost OOF F1     : 0.51721  (threshold=0.5)
Fold 평균          : 0.74017 ± 0.00186


## 4. 비교 요약 & Submission 저장

In [5]:
print('=' * 45)
print(f'  CatBoost OOF AUC : {oof_auc_cat:.5f}')
print(f'  XGBoost  OOF AUC : {oof_auc_xgb:.5f}')
print(f'  EXP-003  OOF AUC : 0.73887  (LightGBM 기준)')
print('=' * 45)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# CatBoost submission
sub['probability'] = test_cat
auc_str = f'{oof_auc_cat:.4f}'.replace('.', '')
fname_cat = f'submission_exp006_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / fname_cat, index=False)

# XGBoost submission
sub['probability'] = test_xgb
auc_str = f'{oof_auc_xgb:.4f}'.replace('.', '')
fname_xgb = f'submission_exp007_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / fname_xgb, index=False)

print(f'CatBoost: {fname_cat}')
print(f'XGBoost : {fname_xgb}')

  CatBoost OOF AUC : 0.74003
  XGBoost  OOF AUC : 0.74016
  EXP-003  OOF AUC : 0.73887  (LightGBM 기준)
CatBoost: submission_exp006_조여진_07400.csv
XGBoost : submission_exp007_조여진_07402.csv


## 5. 자동 실험 기록

In [6]:
def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

# CatBoost 기록
oof_binary = (oof_cat >= 0.5).astype(int)
log_to_leaderboard(
    6, AUTHOR, 'CatBoost',
    f"iter=2000, lr=0.05, depth=6, auto_class_weights=Balanced",
    f1_score(y_train,oof_binary), recall_score(y_train,oof_binary),
    precision_score(y_train,oof_binary), accuracy_score(y_train,oof_binary),
    oof_auc_cat, CV_STR, 'v1', X_train.shape[1], 'auto_class_weights=Balanced',
    'N', None, 'notebooks/06_models_yjcho.ipynb')

# XGBoost 기록
oof_binary = (oof_xgb >= 0.5).astype(int)
log_to_leaderboard(
    7, AUTHOR, 'XGBoost',
    f"n_est=2000, lr=0.05, depth=6, scale_pos_weight={neg_pos_ratio:.2f}",
    f1_score(y_train,oof_binary), recall_score(y_train,oof_binary),
    precision_score(y_train,oof_binary), accuracy_score(y_train,oof_binary),
    oof_auc_xgb, CV_STR, 'v1', X_train.shape[1], f'scale_pos_weight={neg_pos_ratio:.2f}',
    'N', None, 'notebooks/06_models_yjcho.ipynb')

[leaderboard.xlsx] EXP-006 기록 완료 (row 5)
[leaderboard.xlsx] EXP-007 기록 완료 (row 6)


---

## 6. CatBoost v2 — Lossguide + 서브샘플링 (EXP-013)

`grow_policy='Lossguide'`로 leaf-wise 성장 전환. LightGBM 방식과 유사해 표현력이 높아짐.  
`subsample`, `colsample_bylevel` 추가로 과적합 방지.

In [8]:
EXP_NO     = 13
MODEL_NAME = 'CatBoost-v2'

cat_params_v2 = dict(
    iterations        = 2000,
    learning_rate     = 0.05,
    grow_policy       = 'Lossguide',  # leaf-wise (LGB 방식)
    max_leaves        = 64,
    min_data_in_leaf  = 30,
    l2_leaf_reg       = 3,
    subsample         = 0.8,
    colsample_bylevel = 0.8,
    loss_function     = 'Logloss',
    eval_metric       = 'AUC',
    auto_class_weights  = 'Balanced',
    random_seed       = SEED,
    verbose           = False,
    early_stopping_rounds = 100,
)

skf_v2     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_cat_v2  = np.zeros(len(X_train))
test_cat_v2 = np.zeros(len(X_test))
fold_aucs_v2 = []

for fold, (tr_idx, val_idx) in enumerate(skf_v2.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = CatBoostClassifier(**cat_params_v2)
    model.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    fold_aucs_v2.append(auc)
    oof_cat_v2[val_idx]  = val_prob
    test_cat_v2         += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold}  best_iter={model.best_iteration_:4d}  AUC={auc:.5f}')

oof_auc_cat_v2   = roc_auc_score(y_train, oof_cat_v2)
oof_prauc_cat_v2 = average_precision_score(y_train, oof_cat_v2)
oof_f1_cat_v2    = f1_score(y_train, (oof_cat_v2 >= 0.5).astype(int))
print(f'\nCatBoost-v2 OOF ROC-AUC: {oof_auc_cat_v2:.5f}')
print(f'CatBoost-v2 OOF PR-AUC : {oof_prauc_cat_v2:.5f}')
print(f'CatBoost-v2 OOF F1     : {oof_f1_cat_v2:.5f}  (threshold=0.5)')
print(f'EXP-006 대비           : {oof_auc_cat_v2 - oof_auc_cat:+.5f}')

  Fold 1  best_iter= 302  AUC=0.73766
  Fold 2  best_iter= 303  AUC=0.74278
  Fold 3  best_iter= 273  AUC=0.74004
  Fold 4  best_iter= 273  AUC=0.73807
  Fold 5  best_iter= 322  AUC=0.74010

CatBoost-v2 OOF ROC-AUC: 0.73972
CatBoost-v2 OOF PR-AUC : 0.44992
CatBoost-v2 OOF F1     : 0.51712  (threshold=0.5)
EXP-006 대비           : -0.00030


## 7. XGBoost v2 — depth 조정 + gamma 추가 (EXP-014)

In [ ]:
EXP_NO     = 14
MODEL_NAME = 'XGBoost-v2'

xgb_params_v2 = dict(
    n_estimators      = 2000,
    learning_rate     = 0.05,
    max_depth         = 8,       # 6→8 (더 복잡한 패턴 학습)
    min_child_weight  = 30,      # 50→30 (더 세밀한 분할)
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    colsample_bylevel = 0.8,     # 추가: 레벨별 랜덤 피처 선택
    gamma             = 0.1,     # 추가: 분할 최소 손실 감소 (pruning)
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,     # 0.1→1.0 (L2 정규화 강화)
    scale_pos_weight  = neg_pos_ratio,
    eval_metric       = 'auc',
    tree_method       = 'hist',
    random_state      = SEED,
    verbosity         = 0,
    early_stopping_rounds = 100,
)

skf_v2     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_xgb_v2  = np.zeros(len(X_train))
test_xgb_v2 = np.zeros(len(X_test))
fold_aucs_v2 = []

for fold, (tr_idx, val_idx) in enumerate(skf_v2.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(**xgb_params_v2)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    fold_aucs_v2.append(auc)
    oof_xgb_v2[val_idx]  = val_prob
    test_xgb_v2         += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold}  best_iter={model.best_iteration:4d}  AUC={auc:.5f}')

oof_auc_xgb_v2   = roc_auc_score(y_train, oof_xgb_v2)
oof_prauc_xgb_v2 = average_precision_score(y_train, oof_xgb_v2)
oof_f1_xgb_v2    = f1_score(y_train, (oof_xgb_v2 >= 0.5).astype(int))
print(f'\nXGBoost-v2 OOF ROC-AUC: {oof_auc_xgb_v2:.5f}')
print(f'XGBoost-v2 OOF PR-AUC : {oof_prauc_xgb_v2:.5f}')
print(f'XGBoost-v2 OOF F1     : {oof_f1_xgb_v2:.5f}  (threshold=0.5)')
print(f'EXP-007 대비          : {oof_auc_xgb_v2 - oof_auc_xgb:+.5f}')

## 8. Submission 저장 & 실험 기록 (EXP-013 / EXP-014)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# CatBoost-v2 submission
sub_cat_v2 = sub.copy()
sub_cat_v2['probability'] = test_cat_v2
auc_str = f'{oof_auc_cat_v2:.4f}'.replace('.', '')
fname_cat_v2 = f'submission_exp013_{AUTHOR}_{auc_str}.csv'
sub_cat_v2.to_csv(OUT_DIR / fname_cat_v2, index=False)
print(f'CatBoost-v2: {fname_cat_v2}')

# XGBoost-v2 submission
sub_xgb_v2 = sub.copy()
sub_xgb_v2['probability'] = test_xgb_v2
auc_str = f'{oof_auc_xgb_v2:.4f}'.replace('.', '')
fname_xgb_v2 = f'submission_exp014_{AUTHOR}_{auc_str}.csv'
sub_xgb_v2.to_csv(OUT_DIR / fname_xgb_v2, index=False)
print(f'XGBoost-v2 : {fname_xgb_v2}')

print(f'\n=== 결과 요약 ===')
print(f'EXP-006 CatBoost    : {oof_auc_cat:.5f}')
print(f'EXP-013 CatBoost-v2 : {oof_auc_cat_v2:.5f}  ({oof_auc_cat_v2 - oof_auc_cat:+.5f})')
print(f'EXP-007 XGBoost     : {oof_auc_xgb:.5f}')
print(f'EXP-014 XGBoost-v2  : {oof_auc_xgb_v2:.5f}  ({oof_auc_xgb_v2 - oof_auc_xgb:+.5f})')

# CatBoost-v2 기록
oof_bin_cat_v2 = (oof_cat_v2 >= 0.5).astype(int)
log_to_leaderboard(
    13, AUTHOR, 'CatBoost-v2',
    'grow_policy=Lossguide, max_leaves=64, min_data_in_leaf=30, subsample=0.8, colsample_bylevel=0.8',
    f1_score(y_train, oof_bin_cat_v2), recall_score(y_train, oof_bin_cat_v2),
    precision_score(y_train, oof_bin_cat_v2), accuracy_score(y_train, oof_bin_cat_v2),
    oof_auc_cat_v2, CV_STR, 'v1', X_train.shape[1], 'auto_class_weights=Balanced',
    'N', None, 'notebooks/06_models_yjcho.ipynb',
    'Lossguide(leaf-wise)+subsample 추가',
    f'EXP-006 대비 {oof_auc_cat_v2 - oof_auc_cat:+.5f}'
)

# XGBoost-v2 기록
oof_bin_xgb_v2 = (oof_xgb_v2 >= 0.5).astype(int)
log_to_leaderboard(
    14, AUTHOR, 'XGBoost-v2',
    'max_depth=8, min_child_weight=30, colsample_bylevel=0.8, gamma=0.1, reg_lambda=1.0',
    f1_score(y_train, oof_bin_xgb_v2), recall_score(y_train, oof_bin_xgb_v2),
    precision_score(y_train, oof_bin_xgb_v2), accuracy_score(y_train, oof_bin_xgb_v2),
    oof_auc_xgb_v2, CV_STR, 'v1', X_train.shape[1], f'scale_pos_weight={neg_pos_ratio:.2f}',
    'N', None, 'notebooks/06_models_yjcho.ipynb',
    'depth 8, gamma 추가, reg_lambda 강화',
    f'EXP-007 대비 {oof_auc_xgb_v2 - oof_auc_xgb:+.5f}'
)